In [1]:
import pandas as pd

df = pd.read_csv("./imdb_movies.csv")

In [6]:
df

,names,date_x,score,genre,overview,crew,orig_title,status,orig_lang,budget_x,revenue,country
0,Creed III,03/02/2023,73.0,"Drama, Action","After dominating the boxing world, Adonis Cree...","Michael B. Jordan, Adonis Creed, Tessa Thompso...",Creed III,Released,English,75000000.0,2.716167e+08,AU
1,Avatar: The Way of Water,12/15/2022,78.0,"Science Fiction, Adventure, Action",Set more than a decade after the events of the...,"Sam Worthington, Jake Sully, Zoe Saldaña, Neyt...",Avatar: The Way of Water,Released,English,460000000.0,2.316795e+09,AU
2,The Super Mario Bros. Movie,04/05/2023,76.0,"Animation, Adventure, Family, Fantasy, Comedy","While working underground to fix a water main,...","Chris Pratt, Mario (voice), Anya Taylor-Joy, P...",The Super Mario Bros. Movie,Released,English,100000000.0,7.244590e+08,AU
3,Mummies,01/05/2023,70.0,"Animation, Comedy, Family, Adventure, Fantasy","Through a series of unfortunate events, three ...","Óscar Barberán, Thut (voice), Ana Esther Albor...",Momias,Released,"Spanish, Castilian",12300000.0,3.420000e+07,AU
4,Supercell,03/17/2023,61.0,Action,Good-hearted teenager William always lived in ...,"Skeet Ulrich, Roy Cameron, Anne Heche, Dr Quin...",Supercell,Released,English,77000000.0,3.409420e+08,US
...,...,...,...,...,...,...,...,...,...,...,...,...
10173,20th Century Women,12/28/2016,73.0,Drama,"In 1979 Santa Barbara, California, Dorothea Fi...","Annette Bening, Dorothea Fields, Lucas Jade Zu...",20th Century Women,Released,English,7000000.0,9.353729e+06,US
10174,Delta Force 2: The Colombian Connection,08/24/1990,54.0,Action,When DEA agents are taken captive by a ruthles...,"Chuck Norris, Col. Scott McCoy, Billy Drago, R...",Delta Force 2: The Colombian Connection,Released,English,9145817.8,6.698361e+06,US
10175,The Russia House,12/21/1990,61.0,"Drama, Thriller, Romance","Barley Scott Blair, a Lisbon-based editor of R...","Sean Connery, Bartholomew 'Barley' Scott Blair...",The Russia House,Released,English,21800000.0,2.299799e+07,US
10176,Darkman II: The Return of Durant,07/11/1995,55.0,"Action, Adventure, Science Fiction, Thriller, ...",Darkman and Durant return and they hate each o...,"Larry Drake, Robert G. Durant, Arnold Vosloo, ...",Darkman II: The Return of Durant,Released,English,116000000.0,4.756613e+08,US


## Build the Context For Questions

In [7]:
def build_context_string(row):
    context = f"{row['names']} is a {row['genre']} movie released in {row['date_x']} with budget of {row['budget_x']} had revenue {row['revenue']}."
    
    if(description := row['overview']):
        context += f"The movie is all about {description}"
        
    return context

### Helper function for Q&A Pairs

In [8]:
def build_question_string(row, context):
    title = str(row['names'])
    genre = str(row['genre'])
    budget = str(row['budget_x'])
    revenue = str(row['revenue'])
    year = str(row['date_x'])
    description = str(row['overview'])
    
    qa_list = []
    
    #Example question: Budget
    question_1 = f"What is the budget of the movie {title}?"
    answer_1 = budget
    answer_start_1 = context.find(answer_1)
    if answer_start_1 != -1:
        qa_list.append({
            'question': question_1,
            'id': f"{title}_budget",
            'answers': [
                {
                    'text': answer_1,
                    'answer_start': answer_start_1
                }
            ],
            'is_impossible': False
        })
    
    #Example question: Revenue
    question_2 = f"What is the revenue of the movie {title}?"
    answer_2 = revenue
    answer_start_2 = context.find(answer_2)
    if answer_start_2 != -1:
        qa_list.append({
            'question': question_2,
            'id': f"{title}_revenue",
            'answers': [
                {
                    'text': answer_2,
                    'answer_start': answer_start_2
                }
            ],
            'is_impossible': False
        })
        
    #Example question: Genre
    question_3 = f"What is the genre of the movie {title}?"
    answer_3 = genre
    answer_start_3 = context.find(answer_3)
    if answer_start_3 != -1:
        qa_list.append({
            'question': question_3,
            'id': f"{title}_genre",
            'answers': [
                {
                    'text': answer_3,
                    'answer_start': answer_start_3
                }
            ],
            'is_impossible': False
        })
    
    #Example question: Year
    question_4 = f"What is the year of release of the movie {title}?"
    answer_4 = year
    answer_start_4 = context.find(answer_4)
    if answer_start_4 != -1:
        qa_list.append({
            'question': question_4,
            'id': f"{title}_year",
            'answers': [
                {
                    'text': answer_4,
                    'answer_start': answer_start_4
                }
            ],
            'is_impossible': False
        })
    
    question_5 = f"Describe the movie {title}."
    answer_5 = description
    answer_start_5 = context.find(answer_5)
    if answer_start_5 != -1:
        qa_list.append({
            'question': question_5,
            'id': f"{title}_description",
            'answers': [
                {
                    'text': answer_5,
                    'answer_start': answer_start_5
                }
            ],
            'is_impossible': False
        })
        
  
    return qa_list

### Build SQuaD style JSON
We need SQuAD-like formatting if we intend to train an extractive question-answering model using the Hugging Face QA pipeline (e.g., DistilBertForQuestionAnswering fine-tuned in the same way as on SQuAD). 

In other words, the SQuAD-like structure is **required** by the standard HuggingFace QA training scripts 

In [9]:
def build_squad_dataset(df):
  data_list = []
      
  for idx, row in df.iterrows():
      context = build_context_string(row)
      
      qa_list = build_question_string(row, context)
      if(len(qa_list) == 0):
          continue
      
      paragraph_data = {
          'context': context,
          'qas': qa_list
      }
      
      data_list.append({
          'title': row['names'],
          'paragraphs': [paragraph_data]
      })
      
  squad_dict = {
    'title': 'IMDB Movies QA V1',
    'data': data_list
  }

  return squad_dict     
        

In [10]:
squad_data = build_squad_dataset(df)
squad_data

{'title': 'IMDB Movies QA V1',
 'data': [{'title': 'Creed III',
   'paragraphs': [{'context': 'Creed III is a Drama,\xa0Action movie released in 03/02/2023  with budget of 75000000.0 had revenue 271616668.0.The movie is all about After dominating the boxing world, Adonis Creed has been thriving in both his career and family life. When a childhood friend and former boxing prodigy, Damien Anderson, resurfaces after serving a long sentence in prison, he is eager to prove that he deserves his shot in the ring. The face-off between former friends is more than just a fight. To settle the score, Adonis must put his future on the line to battle Damien — a fighter who has nothing to lose.',
     'qas': [{'question': 'What is the budget of the movie Creed III?',
       'id': 'Creed III_budget',
       'answers': [{'text': '75000000.0', 'answer_start': 74}],
       'is_impossible': False},
      {'question': 'What is the revenue of the movie Creed III?',
       'id': 'Creed III_revenue',
       '

In [11]:
from datasets import load_dataset, Dataset, DatasetDict

def squad_dict_to_dataset(squad_dict):
    
    records = []
    
    for entry in squad_dict["data"]:
        title = entry['title']
        for paragraph in entry['paragraphs']:
            context = paragraph['context']
            qas = paragraph['qas']
            for qa in qas:
                question = qa['question']
                id = qa['id']
                is_impossible = qa['is_impossible']
                answers = qa['answers']
                answer_starts = [answer['answer_start'] for answer in answers]
                answer_texts = [answer['text'] for answer in answers]
                
                records.append({
                    'title': title,
                    'context': context,
                    'question': question,
                    'id': id,
                    'is_impossible': is_impossible,
                    'answers': answers,
                    'answer_starts': answer_starts,
                    'answer_texts': answer_texts
                })
    
    return Dataset.from_pandas(pd.DataFrame(records))
            